# Roll D: RFM-tulemuste visualiseerimine

See notebook sisaldab minu individuaalset Roll D väljundit: segmentide jaotus, segmendiprofiil, kümme suurimat VIP-klienti ning soovitused Markole. Sisendiks on Roll C loodud `rfm_viz` DataFrame; kogu järjest käivitatav töövoog asub meeskonna notebook'is.

In [ ]:
# ===== ROLL D: Visualiseerimine ja leiud =====

import pandas as pd
import plotly.express as px

# --- DACA / UrbanStyle värvipalett ---
TEAL = "#009B8D"       # peamine aktsent
NAVY = "#1A1A2E"       # tekst ja teljed
GRAY = "#6B7280"       # tugiinfo
LIGHT_GRAY = "#F3F4F6" # taust
WHITE = "#FFFFFF"      # graafiku taust
BLUE = "#0072B2"       # positiivne / lojaalne
AMBER = "#E69F00"      # hoiatus / potentsiaal
ORANGE = "#D55E00"     # risk

segment_order = [
    "VIP Champions",
    "Loyal",
    "Potential",
    "At Risk",
    "Lost"
]

segment_colors = {
    "VIP Champions": TEAL,
    "Loyal": BLUE,
    "Potential": AMBER,
    "At Risk": ORANGE,
    "Lost": GRAY
}

CHART_WIDTH = 760
CHART_HEIGHT = 430

BASE_LAYOUT = dict(
    plot_bgcolor=WHITE,
    paper_bgcolor=WHITE,
    font=dict(color=NAVY, size=11),
    title=dict(font=dict(color=NAVY, size=17), x=0.02),
)

# --- Lisa RFM tabelile kliendi nimi ---
customer_cols = ["customer_id"]

if "first_name" in df.columns:
    customer_cols.append("first_name")

if "last_name" in df.columns:
    customer_cols.append("last_name")

customer_info = (
    df[customer_cols]
    .drop_duplicates(subset=["customer_id"])
)

rfm_viz = rfm.merge(customer_info, on="customer_id", how="left")

# Kliendi nime loomine: kui nimi puudub, näitame korrektset ID-d
if "first_name" in rfm_viz.columns and "last_name" in rfm_viz.columns:
    rfm_viz["customer_name"] = (
        rfm_viz["first_name"].fillna("") + " " + rfm_viz["last_name"].fillna("")
    ).str.strip()
elif "first_name" in rfm_viz.columns:
    rfm_viz["customer_name"] = rfm_viz["first_name"].fillna("")
else:
    rfm_viz["customer_name"] = ""

rfm_viz["customer_id_clean"] = (
    pd.to_numeric(rfm_viz["customer_id"], errors="coerce")
    .astype("Int64")
    .astype(str)
)

rfm_viz["customer_name"] = rfm_viz["customer_name"].replace("", pd.NA)
rfm_viz["customer_name"] = rfm_viz["customer_name"].fillna(
    "ID " + rfm_viz["customer_id_clean"]
)

rfm_viz["segment"] = pd.Categorical(
    rfm_viz["segment"],
    categories=segment_order,
    ordered=True
)

In [ ]:
segment_counts = (
    rfm_viz["segment"]
    .value_counts()
    .reset_index()
)

segment_counts.columns = ["segment", "klientide_arv"]
segment_counts = segment_counts.sort_values("klientide_arv", ascending=False)

fig1 = px.bar(
    segment_counts,
    x="segment",
    y="klientide_arv",
    color="segment",
    color_discrete_map=segment_colors,
    text="klientide_arv",
    title="UrbanStyle kliendisegmentide jaotus",
    labels={
        "segment": "Kliendisegment",
        "klientide_arv": "Klientide arv"
    },
    category_orders={
        "segment": segment_counts["segment"].tolist()
    },
    width=CHART_WIDTH,
    height=CHART_HEIGHT
)

fig1.update_traces(
    textposition="outside",
    marker_line_width=0,
    textfont=dict(size=11, color=NAVY)
)

fig1.update_layout(
    **BASE_LAYOUT,
    showlegend=False,
    xaxis=dict(
        title="",
        tickfont=dict(size=10),
        tickangle=-10,
        showgrid=False
    ),
    yaxis=dict(
        title="Klientide arv",
        title_font=dict(size=11),
        tickfont=dict(size=10),
        gridcolor=LIGHT_GRAY
    ),
    margin=dict(l=70, r=30, t=70, b=75)
)

fig1.show()

In [ ]:
segment_profile = (
    rfm_viz
    .groupby("segment", observed=True)
    .agg(
        avg_recency=("recency_days", "mean"),
        avg_monetary=("monetary", "mean"),
        customers=("customer_id", "count")
    )
    .reset_index()
)

fig2 = px.scatter(
    segment_profile,
    x="avg_recency",
    y="avg_monetary",
    size="customers",
    color="segment",
    text="segment",
    color_discrete_map=segment_colors,
    category_orders={"segment": segment_order},
    title="RFM segmentide profiil",
    labels={
        "avg_recency": "Päevi viimasest ostust",
        "avg_monetary": "Keskm. kogukulutus",
        "customers": "Klientide arv",
        "segment": "Segment"
    },
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
    size_max=44
)

fig2.update_traces(
    textposition="top center",
    marker=dict(
        opacity=0.78,
        line=dict(width=1, color=WHITE)
    ),
    textfont=dict(size=10, color=NAVY)
)

fig2.update_layout(
    **BASE_LAYOUT,
    showlegend=False,
    xaxis=dict(
        title="Päevi viimasest ostust",
        title_font=dict(size=11),
        tickfont=dict(size=10),
        gridcolor=LIGHT_GRAY
    ),
    yaxis=dict(
        title="Keskm. kogukulutus",
        title_font=dict(size=11),
        tickfont=dict(size=10),
        gridcolor=LIGHT_GRAY,
        tickformat=",.0f"
    ),
    margin=dict(l=75, r=35, t=70, b=70)
)

fig2.show()

In [ ]:
top_vip = (
    rfm_viz[rfm_viz["segment"] == "VIP Champions"]
    .nlargest(10, "monetary")
    .sort_values("monetary", ascending=False)
    .copy()
)

top_vip["monetary_label"] = top_vip["monetary"].map(lambda x: f"{x/1000:.1f}k")

vip_order = top_vip["customer_name"].tolist()

fig3 = px.bar(
    top_vip,
    x="monetary",
    y="customer_name",
    orientation="h",
    text="monetary_label",
    color_discrete_sequence=[TEAL],
    title="Top 10 VIP klienti",
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
    labels={
        "customer_name": "",
        "monetary": "Kogukulutus"
    }
)

fig3.update_traces(
    textposition="inside",
    insidetextanchor="end",
    textfont=dict(color=WHITE, size=11),
    marker_line_width=0,
    hovertemplate="<b>%{y}</b><br>Kogukulutus: %{x:,.2f} EUR<extra></extra>"
)

fig3.update_layout(
    **BASE_LAYOUT,
    showlegend=False,
    xaxis=dict(
        title="Kogukulutus",
        title_font=dict(size=11),
        tickfont=dict(size=10),
        gridcolor=LIGHT_GRAY,
        tickformat=",.0f"
    ),
    yaxis=dict(
        title="",
        tickfont=dict(size=10),
        showgrid=False,
        categoryorder="array",
        categoryarray=vip_order[::-1]
    ),
    margin=dict(l=120, r=35, t=70, b=70)
)

fig3.show()

In [ ]:
# ===== Äritõlgendus Markole =====

total_revenue = rfm_viz["monetary"].sum()

vip_customers = rfm_viz[rfm_viz["segment"] == "VIP Champions"]
at_risk_customers = rfm_viz[rfm_viz["segment"] == "At Risk"]
lost_customers = rfm_viz[rfm_viz["segment"] == "Lost"]
potential_customers = rfm_viz[rfm_viz["segment"] == "Potential"]

vip_count = len(vip_customers)
vip_revenue = vip_customers["monetary"].sum()
vip_revenue_share = vip_revenue / total_revenue * 100

at_risk_count = len(at_risk_customers)
lost_count = len(lost_customers)
potential_count = len(potential_customers)

print(f"""
Äritõlgendus Markole:
UrbanStyle'il on {vip_count} VIP Champions segmenti kuuluvat klienti, kes annavad kokku {vip_revenue_share:.2f}% kogukäibest.
Need kliendid on kõige väärtuslikum segment, sest nad ostavad sageli ja kulutavad palju.
At Risk segmenti kuulub {at_risk_count} klienti ning Lost segmenti {lost_count} klienti, kelle puhul on oluline vähendada kliendikadu.
Potential segmenti kuulub {potential_count} klienti, keda saab sihitud pakkumistega kasvatada lojaalsemateks klientideks.

Soovitused Markole:

VIP programm — paku {vip_count} VIP Champions kliendile eritingimused: varajane juurdepääs 
uutele kollektsioonidele, tasuta saatmine ja personaalsem klienditeenindus. Need kliendid 
annavad suure osa käibest ning nende hoidmine on odavam kui uute klientide leidmine.

Win-back kampaania — saada {at_risk_count} At Risk kliendile personaalne pakkumine järgmise 
30 päeva jooksul, näiteks 15% allahindlus või eripakkumine nende varasema ostukäitumise põhjal. 
Kui nad ei reageeri, võivad nad liikuda Lost segmenti.

Nurture programm — {potential_count} Potential klienti on kasvupotentsiaaliga. 
Lojaalsusprogramm, sihitud pakkumised ja regulaarsed soovitused aitavad neid kasvatada 
Loyal või VIP tasemele.

Lojaalsustaseme kontroll — võrdle RFM segmente ettevõtte olemasoleva loyalty_tier väljaga. 
Kui klient kuulub RFM järgi VIP Champions või Loyal segmenti, aga tema loyalty_tier puudub 
või on madal, tasub talle pakkuda lojaalsusprogrammiga liitumist või kõrgemat taset. 
Nii saab lojaalsusprogramm paremini peegeldada tegelikku ostukäitumist.
""")

# Järeldus

Visualiseeringud muudavad RFM-segmentide suuruse ja väärtuse võrreldavaks. Äriline prioriteet on hoida VIP-kliente, aktiveerida `At Risk` kliendid ning kasvatada `Potential` segmenti lojaalsusprogrammi abil.